# AbdomenNet Local Inference Example

This notebook provides a minimal local tutorial for:

1. validating a user-provided case CSV
2. writing a runnable inference config
3. running checkpoint inference on local, permitted data
4. collecting `pred_npz/test_epoch_0.npz`
5. running a tiny smoke-training example to verify the training entry point

No patient cases or private metadata are distributed with this repository.
Set `ABDOMENNET_CASE_CSV`, `ABDOMENNET_BACKBONE_CKPT`, and `ABDOMENNET_CLASSIFIER_CKPT` for your local environment, or edit the paths below.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import textwrap

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / 'finetune').exists():
    raise RuntimeError('Please launch this notebook from the Acute_abdomen repository root.')

demo_root = repo_root / 'demo' / 'release'
local_cases_dir = demo_root / 'local_cases'

case_csv = Path(os.environ.get('ABDOMENNET_CASE_CSV', str(local_cases_dir / 'cases.csv')))
backbone_ckpt = Path(os.environ.get('ABDOMENNET_BACKBONE_CKPT', 'checkpoints/backbone.pth'))
classifier_ckpt = Path(os.environ.get('ABDOMENNET_CLASSIFIER_CKPT', 'checkpoints/finetuned_classifier.pth'))
python_bin = Path(os.environ.get('PYTHON_BIN', sys.executable))

if not case_csv.is_absolute():
    case_csv = (repo_root / case_csv).resolve()
if not backbone_ckpt.is_absolute():
    backbone_ckpt = (repo_root / backbone_ckpt).resolve()
if not classifier_ckpt.is_absolute():
    classifier_ckpt = (repo_root / classifier_ckpt).resolve()

print('repo_root =', repo_root)
print('case_csv =', case_csv, 'exists =', case_csv.exists())
print('backbone_ckpt =', backbone_ckpt, 'exists =', backbone_ckpt.exists())
print('classifier_ckpt =', classifier_ckpt, 'exists =', classifier_ckpt.exists())
print('python_bin =', python_bin, 'exists =', python_bin.exists())

assert case_csv.exists(), 'Please set ABDOMENNET_CASE_CSV to a valid local CSV.'
assert backbone_ckpt.exists(), 'Please set ABDOMENNET_BACKBONE_CKPT to a valid backbone checkpoint.'
assert classifier_ckpt.exists(), 'Please set ABDOMENNET_CLASSIFIER_CKPT to a valid classifier checkpoint.'
assert python_bin.exists(), 'Please set PYTHON_BIN to a valid Python interpreter.'


In [ ]:
validator = demo_root / 'validate_case_csv.py'
cmd = [str(python_bin), str(validator), str(case_csv)]
print(' '.join(cmd))
result = subprocess.run(cmd, cwd=repo_root, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, 'Local case CSV validation failed'


## Prepare a runnable inference CSV and config

The dataset loader resolves relative image paths against the CSV file location. This notebook also materializes a generated CSV with resolved image paths so the inference config is self-contained.


In [ ]:
df = pd.read_csv(case_csv).copy()
if df.shape[1] < 2:
    raise ValueError('The case CSV must contain one image path column and at least one label column.')

csv_dir = case_csv.parent

def resolve_image_path(value):
    path = Path(str(value).strip())
    return str(path if path.is_absolute() else (csv_dir / path).resolve())

df.iloc[:, 0] = [resolve_image_path(p) for p in df.iloc[:, 0]]
local_csv_abs = demo_root / 'local_cases_abs.csv'
df.to_csv(local_csv_abs, index=False)
num_classes = df.shape[1] - 1
print('generated CSV:', local_csv_abs)
print('num_classes:', num_classes)
display(df.head())

infer_cfg = demo_root / 'abdomennet_local_infer.yaml'
infer_cfg.write_text(textwrap.dedent(f'''
data:
  train_dataset: {json.dumps(str(local_csv_abs))}
  val_dataset: {json.dumps(str(local_csv_abs))}
  test_dataset: {json.dumps(str(local_csv_abs))}
  batch_size: 1
  num_workers: 0
crops:
  global_crops_size: 518
model:
  n_last_blocks_list: [1, 4]
  num_classes: {num_classes}
  mean_slice_feature: false
  mst_type: concate
  backbone_update: false
  use_n_blocks: 4
  use_avgpool: true
  pretrained_weights: {json.dumps(str(backbone_ckpt))}
  ckpt_path: {json.dumps(str(classifier_ckpt))}
  num_decoder_layers: 3
  trans_nhead: 16
  trans_dim_feedforward_ratio: 4
optim:
  loss_type: BCE
  focal_loss_alpha: 0.25
  focal_loss_gamma: 2.0
  step_per_epoch: -1
  epochs: 30
  optimizer: sgd
  sgd_momentum: 0.9
  lr_scheduler: cosine
  weight_decay: 5e-4
  lr: 5e-4
  backbone_lr: 1e-3
  warmup_epochs: 0
  min_lr: 1.0e-06
  clip_grad: 3.0
  patch_embed_lr_mult: 0.2
  layerwise_decay: 1.0
  adamw_beta1: 0.9
  adamw_beta2: 0.999
trainer: EvalTrans25D_Trainer
''').strip() + '
', encoding='utf-8')
print(infer_cfg.read_text())


## Run inference

This command writes predictions to `pred_npz/test_epoch_0.npz`.


In [ ]:
infer_out = demo_root / 'local_infer_output'
infer_out.mkdir(parents=True, exist_ok=True)
cmd = [
    str(python_bin),
    'finetune/run/run_trainer.py',
    '--config-file', str(infer_cfg),
    '--output-dir', str(infer_out),
]
print(' '.join(cmd))
result = subprocess.run(cmd, cwd=repo_root, capture_output=True, text=True)
print('returncode =', result.returncode)
print(result.stdout[-4000:])
print(result.stderr[-4000:])
assert result.returncode == 0, 'Inference failed'


In [ ]:
pred_npz = infer_out / 'pred_npz' / 'test_epoch_0.npz'
assert pred_npz.exists(), pred_npz
print('prediction file:', pred_npz)
obj = __import__('numpy').load(pred_npz, allow_pickle=True)
print(obj.files)
payload = obj[obj.files[0]].item()
print('num rows:', len(payload))
print('sample keys:', list(payload.keys())[:3])


## Tiny smoke-training config

This is not a real training recipe. It only verifies that the public training entry point launches and writes a checkpoint using the local CSV.


In [ ]:
train_cfg = demo_root / 'abdomennet_local_train_smoke.yaml'
train_cfg.write_text(textwrap.dedent(f'''
data:
  train_dataset: {json.dumps(str(local_csv_abs))}
  val_dataset: {json.dumps(str(local_csv_abs))}
  test_dataset: {json.dumps(str(local_csv_abs))}
  batch_size: 1
  num_workers: 0
crops:
  global_crops_size: 518
  global_crops_scale: [1.0, 1.0]
model:
  n_last_blocks_list: [1, 4]
  num_classes: {num_classes}
  mean_slice_feature: false
  mst_type: concate
  backbone_update: false
  use_n_blocks: 4
  use_avgpool: true
  pretrained_weights: {json.dumps(str(backbone_ckpt))}
  num_decoder_layers: 3
  trans_nhead: 16
  trans_dim_feedforward_ratio: 4
optim:
  loss_type: BCE
  focal_loss_alpha: 0.25
  focal_loss_gamma: 2.0
  step_per_epoch: 0
  epochs: 1
  optimizer: sgd
  sgd_momentum: 0.9
  lr_scheduler: cosine
  weight_decay: 5e-4
  lr: 5e-4
  backbone_lr: 1e-3
  warmup_epochs: 0
  min_lr: 1.0e-06
  clip_grad: 3.0
  patch_embed_lr_mult: 0.2
  layerwise_decay: 1.0
  adamw_beta1: 0.9
  adamw_beta2: 0.999
trainer: FinetuneTrans25D_Trainer
''').strip() + '
', encoding='utf-8')
print(train_cfg.read_text())


In [ ]:
train_out = demo_root / 'local_train_smoke_output'
train_out.mkdir(parents=True, exist_ok=True)
cmd = [
    str(python_bin),
    'finetune/run/run_trainer.py',
    '--config-file', str(train_cfg),
    '--output-dir', str(train_out),
]
print(' '.join(cmd))
result = subprocess.run(cmd, cwd=repo_root, capture_output=True, text=True)
print('returncode =', result.returncode)
print(result.stdout[-4000:])
print(result.stderr[-4000:])
assert result.returncode == 0, 'Smoke-training run failed'


In [ ]:
ckpt_dir = train_out / 'checkpoints'
print('checkpoint dir exists =', ckpt_dir.exists())
if ckpt_dir.exists():
    print([p.name for p in ckpt_dir.glob('*.pth')])
